# 05 — Customer Churn Classification

**Goal:** Predict whether a customer will return to make a purchase in Year 2 (2010–2011)
based on their purchasing behavior in Year 1 (2009–2010).

**Target variable:** `is_returning` (1 = returned in Year 2, 0 = did not return)

**Features (derived from Year 1 behavior):**
- Recency, Frequency, Monetary (RFM)
- Average invoice value
- Total items purchased
- Encoded country

**Models:** Logistic Regression, Decision Tree, Random Forest


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('outputs/figures', exist_ok=True)
print(f'Working dir: {os.getcwd()}')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
import json
%matplotlib inline
sns.set_theme(style='whitegrid')
SAVE = lambda name: plt.savefig(f'outputs/figures/{name}', dpi=150, bbox_inches='tight')

## 1. Build Features & Label

In [ ]:
df = pd.read_csv('data/cleaned_retail_customers.csv', parse_dates=['InvoiceDate'])

# Temporal split
SPLIT = pd.Timestamp('2010-12-01')
df_y1 = df[df['InvoiceDate'] < SPLIT]
df_y2 = df[df['InvoiceDate'] >= SPLIT]

y1_customers = set(df_y1['CustomerID'].unique())
y2_customers = set(df_y2['CustomerID'].unique())
print(f"Year-1 customers: {len(y1_customers):,}")
print(f"Year-2 customers: {len(y2_customers):,}")
print(f"Return rate: {len(y1_customers & y2_customers)/len(y1_customers)*100:.1f}%")

In [ ]:
snapshot_y1 = pd.Timestamp('2010-11-30')
feat = df_y1.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate', lambda x: (snapshot_y1 - x.max()).days),
    Frequency       = ('Invoice',     'nunique'),
    Monetary        = ('TotalPrice',  'sum'),
    AvgInvoiceValue = ('TotalPrice',  'mean'),
    TotalItems      = ('Quantity',    'sum'),
    Country         = ('Country',     lambda x: x.mode()[0])
).reset_index()

feat['is_returning'] = feat['CustomerID'].isin(y2_customers).astype(int)

le = LabelEncoder()
feat['Country_enc'] = le.fit_transform(feat['Country'])

print(f"Dataset: {feat.shape}")
print(f"Class balance:\n{feat['is_returning'].value_counts()}")
feat.head()

## 2. Train / Test Split & Scaling

In [ ]:
FEATURES = ['Recency','Frequency','Monetary','AvgInvoiceValue','TotalItems','Country_enc']
X = feat[FEATURES].values
y = feat['is_returning'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

## 3. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1)
}

results, preds, probs = {}, {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    probs[name] = model.predict_proba(X_test)[:,1]
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, preds[name]),  4),
        'Precision': round(precision_score(y_test, preds[name], zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds[name],    zero_division=0), 4),
        'F1':        round(f1_score(y_test, preds[name],        zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, probs[name]),  4),
    }
    print(f"{name:25s}  Acc={results[name]['Accuracy']:.3f}  F1={results[name]['F1']:.3f}  AUC={results[name]['ROC-AUC']:.3f}")

with open('outputs/classification_metrics.json','w') as f:
    json.dump(results, f, indent=2)
pd.DataFrame(results).T

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Return','Return'],
                yticklabels=['Not Return','Return'])
    acc = results[name]['Accuracy']
    ax.set_title(f'{name}\nAccuracy: {acc:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices — Churn Classification', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
SAVE('classification_confusion_matrices.png')
plt.show()

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
line_colors = ['steelblue','coral','seagreen']
for (name, _), color in zip(models.items(), line_colors):
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    auc = results[name]['ROC-AUC']
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random baseline')
ax.set_title('ROC Curves — Classification Models', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10)
plt.tight_layout()
SAVE('classification_roc.png')
plt.show()

## 6. Feature Importance (Random Forest)

In [ ]:
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importances.index, importances.values,
        color=sns.color_palette('viridis', len(FEATURES)))
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
SAVE('classification_feature_importance.png')
plt.show()

## 7. Metrics Comparison

In [ ]:
metrics_df = pd.DataFrame(results).T
metric_cols = ['Accuracy','Precision','Recall','F1','ROC-AUC']
x = np.arange(len(metrics_df))
width = 0.14
colors2 = sns.color_palette('Set2', 5)

fig, ax = plt.subplots(figsize=(12, 6))
for i, (col, color) in enumerate(zip(metric_cols, colors2)):
    ax.bar(x + i*width, metrics_df[col], width, label=col, color=color, edgecolor='white')
ax.set_xticks(x + width*2)
ax.set_xticklabels(metrics_df.index, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Classification Model Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
SAVE('classification_comparison.png')
plt.show()
print("Classification complete!")